In [ ]:
import sys
!{sys.executable} -m pip uninstall -y typing_extensions
!{sys.executable} -m pip install "typing_extensions>=4.13" \
    "pyspark==4.0.1" \
    "kubeflow-spark-api>=2.4.0,<2.5" \
    /home/jovyan/kubeflow-1.0.0-py3-none-any.whl

In [ ]:
import kubeflow
print(kubeflow.__version__)   # 1.0.0

In [ ]:
# restart kernel before running this

import typing_extensions, sys
print("file:", typing_extensions.__file__)
print("has Sentinel:", hasattr(typing_extensions, "Sentinel")) #must be True
print()
for p in sys.path[:8]:
    print(p)

In [ ]:
from kubeflow.spark import SparkClient

client = SparkClient()
try:
    spark = client.connect()
except RuntimeError as e:
    print(e)

In [ ]:
from kubeflow.spark import SparkClient
import kubeflow
print(kubeflow.__version__)      # 1.0.0

In [ ]:
from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.spark import SparkClient, PodTemplateOverride, Driver, Executor

# The SDK defaults to the "default" namespace, where a notebook user has no permissions.
# Naming the profile namespace explicitly is what makes the session land somewhere the
# user can actually create resources. Fixed upstream by kubeflow/sdk#664, which resolves
# the namespace the SDK is running in.
client = SparkClient(
    backend_config=KubernetesBackendConfig(namespace="kubeflow-user-example-com")
)

spark = client.connect(
    driver=Driver(resources={"cpu": "1", "memory": "1Gi"}),
    executor=Executor(
        num_instances=3,
        resources_per_executor={"cpu": "1", "memory": "512Mi"},
    ),
    options=[
        PodTemplateOverride(
            role="driver",
            template={
                # WORKAROUND 1 — Istio.
                # Spark's driver and executors exchange data on ports 7078/7079. With a
                # sidecar in the path, executors register but fail to fetch broadcast
                # blocks, so every task dies. Excluding those ports from interception was
                # not sufficient; the pods have to leave the mesh entirely.
                "metadata": {"labels": {"sidecar.istio.io/inject": "false"}},
                # WORKAROUND 2 — ServiceAccount.
                # The SDK does not set one, so the pod runs as "default", which cannot
                # create executor pods. spark-operator-spark already exists in the
                # namespace with the right RBAC; it just is not used by default.
                "spec": {"serviceAccountName": "spark-operator-spark"},
            },
        ),
        PodTemplateOverride(
            role="executor",
            template={
                # WORKAROUND 1 again — the executors need to be out of the mesh too,
                # since the block transfer that fails is between the two.
                "metadata": {"labels": {"sidecar.istio.io/inject": "false"}},
                # WORKAROUND 3 — Spark Operator 2.3.0 nil-pointer panic.
                # imageOption() dereferences conn.Spec.Executor.Template without a nil
                # check (internal/controller/sparkconnect/options.go). The SDK sets no
                # executor template, so the reconciler panics on every pass and no server
                # pod is ever created. Supplying any template makes the field non-nil.
                # Fixed on the operator's master branch.
                "spec": {"containers": [
                    {"name": "spark-kubernetes-executor", "image": "apache/spark:4.0.4"}
                ]},
            },
        ),
    ]
)

In [ ]:
spark.range(10).show()

In [ ]:
spark.range(1000).createOrReplaceTempView("numbers")
spark.sql("""
    SELECT id % 3 AS grp, COUNT(*) AS n, AVG(id) AS mean
    FROM numbers
    GROUP BY id % 3
    ORDER BY grp
""").show()

In [ ]:
import time
from pyspark.sql.functions import rand, col, when

start = time.time()

SLICES = 400
SAMPLES_PER_SLICE = 500_000
n = SLICES * SAMPLES_PER_SLICE

inside = (
    spark.range(0, n, numPartitions=SLICES)
         .select(rand().alias("x"), rand().alias("y"))
         .select(when(col("x") * col("x") + col("y") * col("y") <= 1.0, 1).otherwise(0).alias("hit"))
         .agg({"hit": "sum"})
         .collect()[0][0]
)

print(f"Pi is roughly {4.0 * inside / n:.6f}")
print(f"{n:,} samples across {SLICES} slices in {time.time() - start:.1f}s")

In [ ]:
for s in client.list_sessions():
    print(f"{s.name}  state={s.state}")

In [ ]:
# remove the sparkconnect session
spark.stop()
for s in client.list_sessions():
    print("deleting", s.name)
    client.delete_session(s.name)